# PD Model Training – Quantum (QSVM / VQC)

**Purpose:** Train hybrid quantum-classical PD models on the same engineered LendingClub features as the XGBoost notebook. We use **PennyLane** (or Qiskit) simulators only (CPU, no real hardware). Models: **QSVM** (quantum kernel) and/or **VQC** (Variational Quantum Classifier / QNN). Compare AUC-ROC, F1, and KS-drift vs the classical XGBoost baseline.

**Input:** `data/credit_risk_pd/LendingClub/processed/lendingclub_engineered.parquet` (run notebook 01 first).

**Dependencies:** `pennylane`, `pennylane-sf` (or default.qubit); optionally `qiskit-aer` for Qiskit path. Install: `pip install pennylane pennylane-sf`.

## 1. Load engineered data (same as notebook 02)

In [1]:
import os
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/leemingloon/ocr-agentic-rag.git"
    REPO_DIR = "ocr-agentic-rag"
    if not os.path.isdir(REPO_DIR):
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    get_ipython().run_line_magic("cd", REPO_DIR)
    get_ipython().system("pip install -q -r notebooks/requirements-notebooks.txt")
    get_ipython().system("pip install -q pennylane pennylane-sf 2>/dev/null || true")
    print("Colab: repo ready. Run the next cells to load data and train the quantum (VQC) model.")
else:
    print("Local run. Ensure you are in the repo root or notebooks/ folder.")

Local run. Ensure you are in the repo root or notebooks/ folder.


In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Find repo root (works when run from repo root or from notebooks/)
ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / "credit_risk").is_dir() and (ROOT / "data").is_dir():
        break
    ROOT = ROOT.parent
if not (ROOT / "credit_risk").is_dir():
    raise RuntimeError('Repo root not found. Run this notebook from the ocr-agentic-rag folder (or notebooks/ subfolder).')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from credit_risk.feature_engineering.common_features import get_feature_names

DATA_PATH = ROOT / "data" / "credit_risk_pd" / "LendingClub" / "processed" / "lendingclub_engineered.parquet"
if not DATA_PATH.exists():
    raise FileNotFoundError("Run 01_lendingclub_feature_engineering.ipynb first.")

df = pd.read_parquet(DATA_PATH)
feature_names = get_feature_names()
X = df[[c for c in feature_names if c in df.columns]].astype(float)
y = df["default"].values
if X.shape[1] != len(feature_names):
    for c in feature_names:
        if c not in X.columns:
            X[c] = 0.0
X = X[feature_names].values
# Remove NaN/Inf so scaling and circuits are well-defined
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

# Scale for quantum circuits (angles / kernel inputs)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=0.0, neginf=0.0)
# Top-6 features by XGBoost importance (better than first-6 for quantum circuits)
n_features_q = min(6, X_scaled.shape[1])
top6_idx = None
try:
    import joblib
    _pkl = ROOT / "models" / "pd" / "pd_model_local_v1.pkl"
    if _pkl.exists():
        _data = joblib.load(_pkl)
        _model = _data.get("model")
        if hasattr(_model, "feature_importances_"):
            imp = _model.feature_importances_
        elif hasattr(_model, "xgb_model"):  # stacked wrapper may hold xgb
            imp = getattr(_model.xgb_model, "feature_importances_", None)
            if imp is None and hasattr(_model, "predict_proba"):
                _model = getattr(_model, "xgb_model", _model)
                imp = getattr(_model, "feature_importances_", None)
        else:
            imp = None
        if imp is not None and len(imp) == X_scaled.shape[1]:
            top6_idx = np.argsort(imp)[-n_features_q:][::-1]
except Exception:
    pass
if top6_idx is None:
    try:
        import xgboost as xgb
        _clf = xgb.XGBClassifier(n_estimators=50, max_depth=4, random_state=42)
        _clf.fit(X, y)
        imp = _clf.feature_importances_
        top6_idx = np.argsort(imp)[-n_features_q:][::-1]
    except Exception:
        top6_idx = np.arange(n_features_q)
X_q = X_scaled[:, top6_idx]
quantum_feature_names = [feature_names[i] for i in top6_idx]

# Same split for quantum (6 feat) and classical (15 feat) comparison
train_idx, val_idx = train_test_split(np.arange(len(y)), test_size=0.2, random_state=42, stratify=y)
X_train, X_val = X_q[train_idx], X_q[val_idx]
y_train, y_val = y[train_idx], y[val_idx]
X_full = df[feature_names].astype(float).values
X_val_full = X_full[val_idx]
print(f"Train {X_train.shape[0]:,} / Val {X_val.shape[0]:,} | Features for quantum: {n_features_q}")

Train 31,828 / Val 7,958 | Features for quantum: 6


## 2. PennyLane – Variational Quantum Classifier (VQC)

**Baseline:** Simple angle-encoding + variational layer and a binary observable. Run on `default.qubit` (CPU simulator).

**Improved (Section 2b):** Data re-uploading, richer encoding, identity-friendly init, class-weighted loss, and stronger optimizer (see frontier/community methods below).

In [ ]:
try:
    import pennylane as qml
    from pennylane import numpy as pnp
    PENNYLANE_AVAILABLE = True
except ImportError:
    PENNYLANE_AVAILABLE = False
    print("Install: pip install pennylane")

if PENNYLANE_AVAILABLE:
    n_qubits = n_features_q
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev)
    def circuit(params, x):
        for i in range(n_qubits):
            qml.RY(x[i] * np.pi, wires=i)
        for i in range(n_qubits):
            qml.RZ(params[i], wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
        for i in range(n_qubits):
            qml.RZ(params[n_qubits + i], wires=i)
        return qml.expval(qml.PauliZ(0))

    def vqc_predict(params, X_batch):
        return np.array([circuit(params, x) for x in X_batch])

    # Simple training loop (gradient descent on binary cross-entropy proxy)
    from scipy.optimize import minimize
    def loss(params):
        pred = np.array([circuit(params, x) for x in X_train[:200]])  # subsample for speed
        pred = (pred + 1) / 2  # map [-1,1] -> [0,1]
        pred = np.clip(pred, 1e-6, 1 - 1e-6)
        return -np.mean(y_train[:200] * np.log(pred) + (1 - y_train[:200]) * np.log(1 - pred))

    init = pnp.random.uniform(0, 2 * np.pi, size=2 * n_qubits)
    res = minimize(loss, init, method="COBYLA", options={"maxiter": 50})
    params_vqc = res.x
    circuit_type = "baseline"
    print('VQC training finished. Params:', params_vqc[:4], '...')
else:
    params_vqc = None
    circuit_type = "baseline"

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score

def eval_binary(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "auc_roc": roc_auc_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

if PENNYLANE_AVAILABLE and params_vqc is not None:
    pred_val = np.array([circuit(params_vqc, x) for x in X_val])
    pred_val = (pred_val + 1) / 2
    pred_val = np.nan_to_num(pred_val, nan=0.5, posinf=1.0, neginf=0.0)
    results_vqc = eval_binary(y_val, pred_val)
    print('VQC (PennyLane) validation:', results_vqc)
else:
    results_vqc = {"auc_roc": 0.5, "f1": 0.0}
    print('VQC skipped (PennyLane not available or training failed).')

VQC (PennyLane) validation: {'auc_roc': 0.49421012000388714, 'f1': 0.21737357259380097}


## 2a. VQC hyperparameter tuning (Optuna)

Tune **n_reupload**, **maxiter**, **n_train** (subsample size), and **init_scale** for the improved VQC to maximize validation AUC before training the final model.


In [ ]:
# Optuna tuning for improved VQC (n_reupload, maxiter, n_train, init_scale)
best_n_reupload = 2
best_maxiter = 200
best_n_train_improved = 1000
best_init_scale = 0.1
if PENNYLANE_AVAILABLE:
    try:
        import optuna
        from scipy.optimize import minimize
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
        def objective(trial):
            n_reupload = trial.suggest_int("n_reupload", 1, 2)
            maxiter = trial.suggest_int("maxiter", 100, 250)
            n_train_improved = trial.suggest_int("n_train_improved", 500, 1200)
            init_scale = trial.suggest_float("init_scale", 0.05, 0.2)
            n_params_per_layer = 4 * n_qubits
            n_params_improved = n_reupload * n_params_per_layer
            dev_t = qml.device("default.qubit", wires=n_qubits)
            @qml.qnode(dev_t, diff_method="backprop")
            def circ(params, x):
                for layer in range(n_reupload):
                    for i in range(n_qubits):
                        qml.RY(x[i] * np.pi, wires=i)
                        qml.RZ(x[i] * np.pi, wires=i)
                    off = layer * n_params_per_layer
                    for i in range(n_qubits):
                        qml.RY(params[off + i], wires=i)
                        qml.RZ(params[off + n_qubits + i], wires=i)
                    for i in range(n_qubits - 1):
                        qml.CNOT(wires=[i, i + 1])
                    for i in range(n_qubits):
                        qml.RY(params[off + 2 * n_qubits + i], wires=i)
                        qml.RZ(params[off + 3 * n_qubits + i], wires=i)
                return qml.expval(qml.sum(*[qml.PauliZ(i) for i in range(n_qubits)]) / n_qubits)
            n_tr = min(n_train_improved, len(X_train))
            X_tr = X_train[:n_tr]
            y_tr = y_train[:n_tr]
            def loss_fn(params):
                pred = np.array([circ (params, x) for x in X_tr])
                pred = (pred + 1) / 2
                pred = np.clip(pred, 1e-6, 1 - 1e-6)
                return -np.mean(scale_pos * y_tr * np.log(pred) + (1 - y_tr) * np.log(1 - pred))
            rng = np.random.default_rng(trial.number)
            init = init_scale * rng.uniform(0, 2 * np.pi, size=n_params_improved)
            res = minimize(loss_fn, init, method="L-BFGS-B", options={"maxiter": maxiter, "disp": False})
            pred_val_t = np.array([circ(res.x, x) for x in X_val])
            pred_val_t = (pred_val_t + 1) / 2
            pred_val_t = np.nan_to_num(pred_val_t, nan=0.5, posinf=1.0, neginf=0.0)
            return roc_auc_score(y_val, pred_val_t)
        study_vqc = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=2))
        study_vqc.optimize(objective, n_trials=6, show_progress_bar=True)
        if study_vqc.best_trial:
            p = study_vqc.best_params
            best_n_reupload = p.get("n_reupload", 2)
            best_maxiter = p.get("maxiter", 200)
            best_n_train_improved = p.get("n_train_improved", 1000)
            best_init_scale = p.get("init_scale", 0.1)
            print("VQC tuning best:", best_n_reupload, best_maxiter, best_n_train_improved, best_init_scale, "AUC", round(float(study_vqc.best_value), 4))
    except Exception as e:
        print("VQC tuning skipped:", e)


## 2b. Improved VQC (frontier & community methods)

Improvements from recent literature and community (Kaggle/PennyLane/arXiv):

1. **Data re-uploading** – Encode data in multiple layers (not just once) to increase expressivity and improve gradient flow (reduces barren-plateau effects) [Pérez-Salinas et al., Schuld et al.].
2. **Richer encoding** – RY + RZ per feature (and optional repeated encoding) for better feature maps; tunable “rotation factor” for kernel expressivity [Scientific Reports 2024].
3. **Identity-friendly / small init** – Initialize variational params small so the circuit starts near identity, avoiding barren plateaus at the first steps [UCL, PennyLane demos].
4. **Class-weighted loss** – Use `scale_pos_weight` (as in XGBoost) for imbalanced default prediction [common in credit risk].
5. **Stronger optimizer & more data** – L-BFGS-B or Adam with more iterations and a larger training subsample (e.g. 500–1000) for better convergence [Qiskit/IBM, empirical comparisons].
6. **Multi-qubit observable** – Measure sum of PauliZ over several qubits for a richer output signal instead of a single qubit.

**Optional:** Use the top-6 features by importance (e.g. from XGBoost `feature_importances_` or `SelectKBest`) instead of the first 6 columns for better quantum feature maps [community/Kaggle].

In [ ]:
# Improved VQC: data re-uploading, richer encoding, small init, class weight, L-BFGS-B
if PENNYLANE_AVAILABLE:
    n_reupload = 2   # data re-uploading layers (literature: 2–3)
    n_var_rots = 2   # RY+RZ per qubit per variational block (before and after entangling)
    # Params: per layer [var_rot_before (2*n_qubits), var_rot_after (2*n_qubits)] = 4*n_qubits per layer
    n_params_per_layer = 4 * n_qubits
    n_params_improved = n_reupload * n_params_per_layer

    @qml.qnode(dev, diff_method="backprop")
    def circuit_improved(params, x):
        for layer in range(n_reupload):
            # Data encoding (re-upload): RY + RZ per qubit (richer than single RY)
            for i in range(n_qubits):
                qml.RY(x[i] * np.pi, wires=i)
                qml.RZ(x[i] * np.pi, wires=i)
            # Variational block (identity-friendly when params ≈ 0)
            off = layer * n_params_per_layer
            for i in range(n_qubits):
                qml.RY(params[off + i], wires=i)
                qml.RZ(params[off + n_qubits + i], wires=i)
            for i in range(n_qubits - 1):
                qml.CNOT(wires=[i, i + 1])
            for i in range(n_qubits):
                qml.RY(params[off + 2 * n_qubits + i], wires=i)
                qml.RZ(params[off + 3 * n_qubits + i], wires=i)
        # Multi-qubit observable: average PauliZ (richer than single qubit)
        return qml.expval(qml.sum(*[qml.PauliZ(i) for i in range(n_qubits)]) / n_qubits)

    scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    n_train_improved = min(best_n_train_improved if "best_n_train_improved" in dir() else 1000, len(X_train))
    X_tr = X_train[:n_train_improved]
    y_tr = y_train[:n_train_improved]

    def loss_improved(params):
        pred = np.array([circuit_improved(params, x) for x in X_tr])
        pred = (pred + 1) / 2
        pred = np.clip(pred, 1e-6, 1 - 1e-6)
        # Class-weighted BCE (imbalanced credit default)
        pos_w = scale_pos
        return -np.mean(pos_w * y_tr * np.log(pred) + (1 - y_tr) * np.log(1 - pred))

    # Identity-friendly init (use tuned init_scale from 2a)
    rng = np.random.default_rng(42)
    init_scale = best_init_scale if "best_init_scale" in dir() else 0.1
    init_improved = init_scale * rng.uniform(0, 2 * np.pi, size=n_params_improved)

    from scipy.optimize import minimize
    maxiter = best_maxiter if "best_maxiter" in dir() else 200
    res_improved = minimize(
        loss_improved, init_improved, method="L-BFGS-B",
        options={"maxiter": maxiter, "disp": False},
    )
    params_vqc_improved = res_improved.x
    print("Improved VQC training finished. Loss:", round(float(res_improved.fun), 4))

    pred_val_improved = np.array([circuit_improved(params_vqc_improved, x) for x in X_val])
    pred_val_improved = (pred_val_improved + 1) / 2
    pred_val_improved = np.nan_to_num(pred_val_improved, nan=0.5, posinf=1.0, neginf=0.0)
    results_vqc_improved = eval_binary(y_val, pred_val_improved)
    print("Improved VQC (PennyLane) validation:", results_vqc_improved)

    if results_vqc_improved["auc_roc"] >= results_vqc["auc_roc"]:
        results_vqc = results_vqc_improved
        pred_val = pred_val_improved
        params_vqc = params_vqc_improved
        circuit_type = "improved"
        print("Using improved VQC for comparison with XGBoost.")
else:
    print("PennyLane not available; improved VQC skipped.")

C:\Users\leemi\AppData\Local\Temp\ipykernel_13124\3343509995.py:47: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res_improved = minimize(


## 2c. Optional: QSVM (quantum kernel + SVM)

Use a **quantum feature map** to build a kernel matrix, then train a classical SVM on it (kernel alignment and tunable feature maps often improve QSVM [Scientific Reports, IEEE]). We use a small subsample for the kernel to keep runtime tractable.

In [ ]:
# Optional QSVM: quantum kernel + SVC (subsample for speed)
results_qsvm = None
if PENNYLANE_AVAILABLE:
    try:
        from sklearn.svm import SVC
        n_qsvm = min(800, len(X_train))
        X_tr_k = X_train[:n_qsvm]
        y_tr_k = y_train[:n_qsvm]
        dev_k = qml.device("default.qubit", wires=n_qubits)
        @qml.qnode(dev_k)
        def kernel_circuit(x1, x2):
            for i in range(n_qubits):
                qml.RY(x1[i] * np.pi, wires=i)
                qml.RZ(x1[i] * np.pi, wires=i)
            for i in range(n_qubits):
                qml.RY(-x2[i] * np.pi, wires=i)
                qml.RZ(-x2[i] * np.pi, wires=i)
            return qml.probs(wires=list(range(n_qubits)))
        def kernel_fn(x1, x2):
            return kernel_circuit(x1, x2)[0]
        K_train = qml.kernels.square_kernel_matrix(X_tr_k, kernel_fn)
        K_train = np.nan_to_num(K_train, nan=0.0, posinf=1.0)
        clf_qsvm = SVC(kernel="precomputed", C=1.0, class_weight="balanced", random_state=42)
        clf_qsvm.fit(K_train, y_tr_k)
        K_val = qml.kernels.kernel_matrix(X_val, X_tr_k, kernel_fn)
        K_val = np.nan_to_num(K_val, nan=0.0, posinf=1.0)
        p_qsvm = clf_qsvm.decision_function(K_val)
        p_qsvm = (p_qsvm - p_qsvm.min()) / (p_qsvm.max() - p_qsvm.min() + 1e-8)
        results_qsvm = eval_binary(y_val, p_qsvm)
        print("QSVM (quantum kernel + SVC) validation:", results_qsvm)
    except Exception as e:
        print("QSVM skipped:", e)

## 3. Classical baseline (benchmark only)

Load the XGBoost model from notebook 02 to report AUC/F1 **comparison** and uplift %. The quantum (VQC) model remains the main output; this section is for benchmarking only.

In [ ]:
import joblib

model_path = ROOT / "models" / "pd" / "pd_model_local_v1.pkl"
results_xgb = None
if model_path.exists():
    try:
        data = joblib.load(model_path)
        xgb_model = data["model"]
        p_xgb = xgb_model.predict_proba(X_val_full)[:, 1]
        p_xgb = np.nan_to_num(p_xgb, nan=0.5)
        results_xgb = eval_binary(y_val, p_xgb)
        print('XGBoost (classical, 15-feat) validation:', results_xgb)
    except Exception as e:
        print('Failed to load/run XGBoost model:', e)
        results_xgb = None
if results_xgb is None:
    try:
        import xgboost as xgb
        scale = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
        clf = xgb.XGBClassifier(max_depth=4, n_estimators=50, scale_pos_weight=scale, random_state=42)
        clf.fit(X_train, y_train)
        p_xgb = clf.predict_proba(X_val)[:, 1]
        p_xgb = np.nan_to_num(p_xgb, nan=0.5)
        results_xgb = eval_binary(y_val, p_xgb)
        print('XGBoost (6-feature baseline) validation:', results_xgb)
    except ImportError:
        results_xgb = {"auc_roc": 0.5, "f1": 0.0}
        print('XGBoost not installed. Install with: pip install xgboost. Skipping classical comparison.')
    except Exception as e:
        results_xgb = {"auc_roc": 0.5, "f1": 0.0}
        print('XGBoost baseline failed:', e)

if results_xgb is not None:
    print('\nUplift vs classical (VQC - XGB): AUC', round(results_vqc['auc_roc'] - results_xgb['auc_roc'], 4), 'F1', round(results_vqc['f1'] - results_xgb['f1'], 4))

XGBoost (classical, 15-feat) validation: {'auc_roc': 0.6842322123804148, 'f1': 0.32768623810628916}

Uplift vs classical (VQC - XGB): AUC -0.19 F1 -0.1103


## 4. Save quantum model (main output)

**Primary output:** Always save the **quantum (VQC) model** to `pd_quantum_vqc_v1.pkl` when training succeeds. This is the main artifact for PD prediction; classical and blend are for benchmarking only. NISQ note: in production, consider noise mitigation and qubit scaling; simulators are for prototyping.

In [ ]:
# Quantum–classical blend: tune alpha on validation
blend_alpha_opt = 0.5
if results_xgb is not None and "pred_val" in dir() and pred_val is not None:
    best_auc, best_alpha = 0.0, 0.5
    for a in np.linspace(0.1, 0.9, 9):
        p_blend = a * np.asarray(pred_val).ravel() + (1 - a) * np.asarray(p_xgb).ravel()
        p_blend = np.clip(p_blend, 0.0, 1.0)
        r = eval_binary(y_val, p_blend)
        if r["auc_roc"] > best_auc:
            best_auc, best_alpha = r["auc_roc"], a
    blend_alpha_opt = best_alpha
    p_blend_final = blend_alpha_opt * np.asarray(pred_val).ravel() + (1 - blend_alpha_opt) * np.asarray(p_xgb).ravel()
    results_blend = eval_binary(y_val, p_blend_final)
    print("Blend (quantum + XGB) best alpha:", round(blend_alpha_opt, 2), "| Val AUC:", round(results_blend["auc_roc"], 4), "F1:", round(results_blend["f1"], 4))
else:
    print("Blend skipped (need both quantum pred_val and XGBoost p_xgb).")

In [ ]:
MODEL_DIR = ROOT / "models" / "pd"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
# Primary output: always save the quantum (VQC) model when we have it
if PENNYLANE_AVAILABLE and params_vqc is not None:
    joblib.dump({
        "params": params_vqc,
        "n_qubits": n_qubits,
        "n_params": len(params_vqc),
        "circuit_type": circuit_type if "circuit_type" in dir() else "baseline",
        "scaler": scaler,
        "feature_names": quantum_feature_names if "quantum_feature_names" in dir() else feature_names[:n_features_q],
        "backend": "pennylane.default.qubit",
    }, MODEL_DIR / "pd_quantum_vqc_v1.pkl")
    print('Saved quantum (VQC) model to models/pd/pd_quantum_vqc_v1.pkl')
else:
    print('No quantum model to save: run Section 2 (VQC) to train the quantum model.')
print('Done. Use eval_runner for quantum PD; classical XGBoost is benchmark only.')

Saved VQC params to models/pd/pd_quantum_vqc_v1.pkl
Done. Use eval_runner with PDModel for classical; extend run_model for quantum backend if needed.
